# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do NOT treat as dict for subscripting/iteration

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Authors: {', '.join([str(a) for a in getattr(metadata, 'author', [])])}")
print(f"Croissant Version: {getattr(metadata, 'conformsTo', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets
print("Available record sets (referenced by @id):\n")
for rs in metadata.recordSets:
    print(f"- @id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}\n")
    print("  Fields (with @id and name):")
    for field in rs.fields:
        print(f"    - @id: {field.id} | name: {field.name} | type: {field.dataType if hasattr(field, 'dataType') else 'N/A'}")
    print()
    # For brevity, show only first one or two record sets. You may remove below break in a real use case.
    # break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set(s) by @id — replace these with actual record set @id values from previous cell's output
# We'll extract ALL record sets found in the metadata.
record_sets = [rs.id for rs in metadata.recordSets]
print("Extracting these record sets:", record_sets)

dataframes = {}

for record_set_id in record_sets:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} rows and columns: {df.columns.tolist()}\n")
    else:
        print("No records found for this record set.\n")

# For further analysis, let's focus on the main protein record set
# Example: Guessing a main table id (commonly 'cr:proteins' or similar, inspect above output)
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"First few rows of main record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets with data found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section applies to the main record set extracted above.

In [ ]:
# Select one numeric field for analysis (update with actual field @id from previous overview)
import numpy as np

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]

    # List numeric-like columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Attempt to coerce numeric types
        for c in df.columns:
            with np.errstate(invalid='ignore'):
                df[c + "_numeric"] = pd.to_numeric(df[c], errors='coerce')
        numeric_cols = [col for col in df.columns if col.endswith('_numeric')]
    print("Numeric fields detected:", numeric_cols)

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in '{main_record_set_id}' where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        # List possible grouping fields (non-numeric columns)
        group_candidates = [c for c in df.columns if not c.endswith('_numeric') and c != numeric_field]
        group_field = None
        for c in group_candidates:
            if pd.api.types.is_string_dtype(df[c]) and df[c].nunique() < 50 and df[c].nunique() > 1:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by '{group_field}':")
            display(grouped_df)
        else:
            print("No suitable group-by field found!")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No data available for main record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and not dataframes[main_record_set_id].empty and numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set '{main_record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if group_field exists)
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=dataframes[main_record_set_id], x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No sufficient numeric data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading the FAIR^2 dataset via the Croissant schema using `mlcroissant`. We explored the dataset's record sets and fields (referenced always by their `@id`), loaded data into DataFrames, performed basic filtering and normalization, and visualized numeric data fields. For detailed biological or domain analysis, proceed with field-specific hypotheses or utilize more advanced bioinformatics tools. This workflow, using the power of Croissant's standardized metadata, enables reproducible and robust dataset exploration.
